# Criando tabela bronze apartir de arquivo JSON

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *

@dp.table(
    comment="Tabela bronze com dados brutos de livros do arquivo JSON"
)
def livros_pipe():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.inferColumnTypes", "true")
            .load("/Volumes/dbportifolio/pipeline/files/")
    )

In [0]:
# Query de verificação - expandindo campos do endereço
@dp.temporary_view()
def livros_expandido():
    return (
        spark.readStream.table("livros_pipe")
            .select(
                col("*"),
                col("endereco.estado").alias("estado"),
                col("endereco.cidade").alias("cidade"),
                col("endereco.bairro").alias("bairro"),
                col("endereco.rua").alias("rua"),
                col("endereco.numero").alias("numero"),
                col("endereco.cep").alias("cep")
            )
            .drop("endereco")
    )

# Lendo camada bronze e gerando três tabelas pratas, uma de endereços, autores e uma de vendas de livros

In [0]:
from pyspark.sql.window import Window

@dp.materialized_view(
    comment="Tabela silver com endereços únicos normalizados"
)
def enderecos_pipe():
    return (
        spark.read.table("livros_pipe")
            .select("endereco")
            .distinct()
            .select(
                row_number().over(Window.orderBy("endereco")).alias("id"),
                col("endereco.estado").alias("estado"),
                col("endereco.cidade").alias("cidade"),
                col("endereco.bairro").alias("bairro"),
                col("endereco.rua").alias("rua"),
                col("endereco.numero").alias("numero"),
                col("endereco.cep").alias("cep")
            )
    )

In [0]:
from pyspark.sql.window import Window

@dp.materialized_view(
    comment="Tabela silver com autores únicos normalizados"
)
def autores_pipe():
    return (
        spark.read.table("livros_pipe")
            .select("autor")
            .distinct()
            .select(
                row_number().over(Window.orderBy("autor")).alias("id"),
                col("autor")
            )
    )

In [0]:
@dp.table(
    comment="Tabela silver com vendas de livros relacionadas com autores e endereços"
)
def vendas_livros_pipe():
    livros = spark.readStream.table("livros_pipe")
    autores = spark.read.table("autores_pipe")
    enderecos = spark.read.table("enderecos_pipe")
    
    # Join com autores
    df = livros.join(autores, livros.autor == autores.autor, "inner")
    
    # Join com endereços
    df = df.join(
        enderecos,
        (livros.endereco.estado == enderecos.estado) &
        (livros.endereco.cidade == enderecos.cidade) &
        (livros.endereco.bairro == enderecos.bairro) &
        (livros.endereco.rua == enderecos.rua) &
        (livros.endereco.numero == enderecos.numero) &
        (livros.endereco.cep == enderecos.cep),
        "inner"
    )
    
    # Selecionar colunas excluindo endereco e autor duplicados
    cols_to_keep = [col for col in livros.columns if col not in ["endereco", "autor"]]
    
    return df.select(
        *[livros[c] for c in cols_to_keep],
        autores.id.alias("autor_id"),
        enderecos.id.alias("endereco_id")
    )

# Por fim vamos criar duas tabelas ouro, uma de autor mais vendidos e qtd de livros vendidos por cidade

In [0]:
@dp.materialized_view(
    comment="Tabela gold com o autor mais vendido"
)
def autor_mais_vendido_pipe():
    vendas = spark.read.table("vendas_livros_pipe")
    autores = spark.read.table("autores_pipe")
    
    return (
        vendas.join(autores, vendas.autor_id == autores.id, "inner")
            .groupBy("autor")
            .agg(sum("quantidade").alias("total_vendas"))
            .orderBy(col("total_vendas").desc())
            .limit(1)
    )

In [0]:
@dp.materialized_view(
    comment="Tabela gold com total de vendas agrupadas por cidade"
)
def vendas_cidades_pipe():
    vendas = spark.read.table("vendas_livros_pipe")
    enderecos = spark.read.table("enderecos_pipe")
    
    return (
        vendas.join(enderecos, vendas.endereco_id == enderecos.id, "inner")
            .groupBy("cidade")
            .agg(sum("quantidade").alias("total_vendas"))
    )